# pack code

In [ ]:
import os
import json

# 設定專案路徑
project_path = r'C:\Users\slab\Desktop\Slab Project\Route Planner'
output_file = 'Route_Planner_Full_Code.txt'

# 定義要抓取的副檔名
target_extensions = ('.py', '.ipynb')

print(f"正在掃描專案: {project_path}")

if not os.path.exists(project_path):
    print(f"找不到路徑: {project_path}，請檢查路徑是否正確。")
else:
    with open(output_file, 'w', encoding='utf-8') as f:
        for root, dirs, files in os.walk(project_path):
            # 排除掉不需要的資料夾（如虛擬環境、快取、Git）
            if any(part in root for part in ['.venv', '__pycache__', '.git', 'legacy']):
                continue
                
            for file in files:
                if file.endswith(target_extensions):
                    full_path = os.path.join(root, file)
                    rel_path = os.path.relpath(full_path, project_path)
                    
                    # 寫入檔案標記與分隔線
                    f.write(f"\n\n{'='*50}\n")
                    f.write(f"FILE_PATH: {rel_path}\n")
                    f.write(f"{'='*50}\n\n")
                    
                    try:
                        if file.endswith('.ipynb'):
                            # 針對 Jupyter Notebook 提取程式碼區塊，避免雜訊太重
                            with open(full_path, 'r', encoding='utf-8', errors='ignore') as jupyter_f:
                                nb_content = json.load(jupyter_f)
                                for cell in nb_content.get('cells', []):
                                    if cell['cell_type'] == 'code':
                                        f.write("".join(cell['source']) + "\n")
                        else:
                            # 一般 .py 檔案直接讀取
                            with open(full_path, 'r', encoding='utf-8', errors='ignore') as code_f:
                                f.write(code_f.read())
                    except Exception as e:
                        f.write(f"讀取失敗: {str(e)}")

    print(f"成功！已掃描所有檔案（包含 routing_map 內容）。")
    print(f"請將生成的 {os.path.abspath(output_file)} 上傳到 AI Studio。")

# Debug for new pipeline

## build_aoi

In [1]:
from pathlib import Path
import routing_map
from routing_map import build_aoi, RoutingMapConfig
from routing_map.config import AoiConfig, LandConfig
from routing_map.ring_types import RingBuildConfig
from routing_map import cache_utils


cfg = RoutingMapConfig(
    aoi=AoiConfig(
        # 全地圖範圍
        # bbox_ll=((-179.999, -85, 179.999, 85)), -26.350271, 51.260190
        bbox_ll=((42.251401, -10.787699, 51.260190, -26.350271)),
    ),
    land=LandConfig(
        shp_path=Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\GSHHS\GSHHS_shp\h\GSHHS_h_L1.shp"),
        buffer_km=20.0,
        avoid_km= 1,
        collision_safety_km=0.5,
    ),
        rings=RingBuildConfig(
        clearance_m=10000.0,      # 1 km：你 land.avoid_km 的概念
        ring_sample_km=5.0,      # 采樣間距（先保守）
        taut_window_size=80,
        taut_max_tries=8,
        min_ring_length_km=20.0,

        taut_use_clearance_buffer= True,
        taut_collision_buffer_m= 5000,
    ),
    
)


CACHE_DIR = r"C:\Users\slab\Desktop\Slab Project\Route Planner\aoi_cache"
CACHE_TAG = "global"  # 你可以改成 "taiwan" / "world" / "dateline" 都行

out = cache_utils.get_out(cfg, cache_dir=CACHE_DIR, use_cache=True)



In [3]:
import folium
import webbrowser
import os
from routing_map.viz_layers import (
    make_base_map,
    add_ring_layers,
    add_connector_layers,
    add_sea_layers,
    finalize_map
)

# 1. 取得範圍並建立底圖
bbox_ll = out["bbox_ll"]
print(f"正在為範圍 {bbox_ll} 建立視覺化地圖...")

# 使用你專案內建的底圖產生器 (會自動處理跨日界線與中心點)
m = make_base_map(bbox_ll=bbox_ll, zoom_start=8)

# 2. 加入 Sea Layers (深海航網)
# 這裡設定 node_sample=None 表示在小範圍內顯示所有節點，不進行抽樣
add_sea_layers(
    m, out, 
    node_sample=None, 
    max_edges=5000, 
    show=True, 
    bbox_ll=bbox_ll
)

# 3. 加入 Ring Layers (E-Ring 紅線, T-Ring 綠線, 以及對應節點)
# E_nodes 會是藍色點, T_nodes 會是綠色點
add_ring_layers(
    m, out,
    e_node_sample=None, 
    t_node_sample=None,
    max_e_edges=10000,
    max_t_edges=10000,
    show=True,
    bbox_ll=bbox_ll
)

# 4. 加入 Connector Layers (E<->T 轉換邊, 以及 T-gate->Sea 橘色接入邊)
add_connector_layers(
    m, out,
    show_et=True,
    show_tgate_sea=True,
    max_et=5000,
    max_tgate=5000,
    show=True,
    bbox_ll=bbox_ll
)

# 5. 儲存並開啟 HTML
output_file = "aoi_et_rings_debug.html"
finalize_map(m, html_path=output_file)

full_path = os.path.abspath(output_file)
print(f"地圖已儲存至: {full_path}")
webbrowser.open(f"file://{full_path}")

正在為範圍 (42.251401, -26.350271, 51.26019, -10.787699) 建立視覺化地圖...
地圖已儲存至: c:\Users\slab\Desktop\Slab Project\Route Planner\aoi_et_rings_debug.html


True

## build_G

In [2]:
from routing_map.pipeline import GraphConfig
from routing_map import cache_utils

# 1. 設定建圖參數 (GraphConfig)
# 這裡建議把 bbox_ll 傳進去，確保圖資範圍正確
graph_cfg = GraphConfig(
    bbox_ll=out["bbox_ll"],
    max_sea_edges=None,
    max_ring_edges=None,
    weight_unit="km",
)

# 2. 準備建圖用的參數字典
# 這些參數會影響圖資的指紋，決定是否命中快取
graph_build_args = dict(
    include_sea=graph_cfg.include_sea,
    include_rings=graph_cfg.include_rings,
    include_et=graph_cfg.include_et,
    include_tgate_sea=graph_cfg.include_tgate_sea,
    max_sea_edges=graph_cfg.max_sea_edges,
    max_ring_edges=graph_cfg.max_ring_edges,
    weight_unit=graph_cfg.weight_unit,
    bbox_ll=out["bbox_ll"],
)

# 3. 執行建圖 (這一步才會產生 G_global.pkl.gz)
print("正在從 out 建構導航圖資 (G_base)...")
G_base = cache_utils.get_graph(
    out,
    cfg=cfg,
    graph_build_args=graph_build_args,
    cache_dir=CACHE_DIR,
    use_cache=True, # 設為 True，算完會自動存成 G_global.pkl.gz
)

print(f"建圖完成！節點數: {G_base.number_of_nodes()}, 邊數: {G_base.number_of_edges()}")

正在從 out 建構導航圖資 (G_base)...
建圖完成！節點數: 1447, 邊數: 1665
